In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

PARQUET_DIR = Path("parquet_output")

parquet_files = {
    "crashes_slim": PARQUET_DIR / "crashes_slim.parquet",
}

for name, path in parquet_files.items():
    print("=" * 80)
    print(f"{name}: {path}")
    
    schema = pq.read_schema(path)
    columns = schema.names
    
    print(f"columns_count = {len(columns)}")
    print()
    
    for i, col in enumerate(columns, start=1):
        print(f"{i:02d}. {col}")
    
    print()

crashes: parquet_output/crashes.parquet
columns_count = 35

01. crash_uid
02. ADDITIONAL_ATTR
03. CHILDREN_ATTR
04. DATE_TIME
05. EMTP_NUMBER
06. EM_NUMBER
07. EM_TYPE
08. HOUSE
09. LAT
10. LNG
11. LIGHT_TYPE
12. LOST_AMOUNT
13. MT_RATE
14. NP
15. PERS_AMOUNT
16. PLACE
17. ROAD_KM_M
18. ROADS
19. ROAD_SIGN
20. ROAD_TYPE
21. SCHEMA_ID
22. STREET
23. STREET_SIGN
24. SUFFER_AMOUNT
25. TRAFFIC_AREA_STATE
26. VEH_AMOUNT
27. POINT_TYPE
28. POINT_LNG
29. POINT_LAT
30. CLOUDS_JSON
31. DEFECTS_JSON
32. MOTION_INFLUENCES_JSON
33. RD_CONSTR_HERES_JSON
34. RD_CONSTR_THERES_JSON
35. REGIONS_JSON

vehicles: parquet_output/vehicles.parquet
columns_count = 14

01. crash_uid
02. vehicle_index
03. N
04. COLOR
05. ESCAPE
06. MARK
07. MODEL
08. OKFS
09. OKOPF
10. PROD_TYPE
11. RUDDER_TYPE
12. VEHICLE_YEAR
13. DAMAGE_DISPOSITION_JSON
14. TECH_FAILURE_TYPE_JSON

persons: parquet_output/persons.parquet
columns_count = 16

01. crash_uid
02. person_scope
03. vehicle_index
04. person_index
05. N
06. PART_TYPE
0

In [2]:
import pandas as pd

crashes = pd.read_parquet("parquet_output/crashes.parquet")

coord_check = crashes[["LAT", "LNG", "POINT_LAT", "POINT_LNG"]].copy()

print("Missing values:")
print(coord_check.isna().sum())

coord_check["lat_diff"] = (coord_check["LAT"] - coord_check["POINT_LAT"]).abs()
coord_check["lng_diff"] = (coord_check["LNG"] - coord_check["POINT_LNG"]).abs()

print("\nLat diff:")
print(coord_check["lat_diff"].describe())

print("\nLng diff:")
print(coord_check["lng_diff"].describe())

print("\nRows with noticeable coordinate mismatch:")
display(coord_check[
    (coord_check["lat_diff"] > 1e-6) | (coord_check["lng_diff"] > 1e-6)
].head(20))

Missing values:
LAT          162
LNG          154
POINT_LAT    163
POINT_LNG    163
dtype: int64

Lat diff:
count    1.616260e+06
mean     1.886194e+01
std      1.970115e+01
min      0.000000e+00
25%      5.883391e+00
50%      1.302990e+01
75%      2.588369e+01
max      2.484037e+02
Name: lat_diff, dtype: float64

Lng diff:
count    1.616260e+06
mean     1.886194e+01
std      1.970115e+01
min      0.000000e+00
25%      5.883391e+00
50%      1.302990e+01
75%      2.588369e+01
max      2.484037e+02
Name: lng_diff, dtype: float64

Rows with noticeable coordinate mismatch:


,LAT,LNG,POINT_LAT,POINT_LNG,lat_diff,lng_diff
1,37.400000,57.100000,57.100000,37.400000,19.700000,19.700000
2,61.561944,54.083611,54.083611,61.561944,7.478333,7.478333
3,94.457500,51.708333,51.708333,94.457500,42.749167,42.749167
4,35.503300,56.029400,56.029400,35.503300,20.526100,20.526100
5,37.608056,55.689444,55.689444,37.608056,18.081388,18.081388
6,102.165556,54.114722,54.114722,102.165556,48.050834,48.050834
7,131.135300,43.585300,43.585300,131.135300,87.550000,87.550000
8,36.061111,52.985833,52.985833,36.061111,16.924722,16.924722
9,20.361400,54.452200,54.452200,20.361400,34.090800,34.090800
10,142.000000,46.000000,46.000000,142.000000,96.000000,96.000000


In [3]:
coord_check["lat_vs_lng_field_diff"] = (coord_check["POINT_LAT"] - coord_check["LNG"]).abs()
coord_check["lng_vs_lat_field_diff"] = (coord_check["POINT_LNG"] - coord_check["LAT"]).abs()

print("POINT_LAT vs LNG:")
print(coord_check["lat_vs_lng_field_diff"].describe())

print("\nPOINT_LNG vs LAT:")
print(coord_check["lng_vs_lat_field_diff"].describe())

print("\nBad swapped matches:")
display(coord_check[
    (coord_check["lat_vs_lng_field_diff"] > 1e-6) |
    (coord_check["lng_vs_lat_field_diff"] > 1e-6)
].head(20))

POINT_LAT vs LNG:
count    1616260.0
mean           0.0
std            0.0
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0
Name: lat_vs_lng_field_diff, dtype: float64

POINT_LNG vs LAT:
count    1.616260e+06
mean     8.352617e-12
std      1.699800e-09
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.000000e-07
Name: lng_vs_lat_field_diff, dtype: float64

Bad swapped matches:


,LAT,LNG,POINT_LAT,POINT_LNG,lat_diff,lng_diff,lat_vs_lng_field_diff,lng_vs_lat_field_diff
